## **Expiry-Day Liquidity Concentration in Indian Index Derivatives**

### **Notebook: 04: Corrected Difference-in-Differences (NSE and BSE)**

---

**Scope:** True-expiry symbol-month DiD. Within-exchange: Bank Nifty vs Nifty (NSE) and Bankex vs Sensex (BSE). Pooled: six indices, symbol and month fixed effects, exact randomization inference over the 15 treatment assignments. Reproduce: `python -m scripts.did_monthly_true_expiry`, `python -m scripts.cross_exchange_did`.

---

### Pipeline sequence

1. **01_data_collection.ipynb** -- index OHLCV, F&O Bhavcopy archive (NOTE: the rule-based expiry calendar it builds is NOT used for inference)
2. **02_feature_engineering_and_fo_construction.ipynb** -- technical features (true-calendar expiry features) and the true-expiry index panel
3. **03_weekly_options_discontinuation_identification.ipynb** -- true expiry dates per index and a calendar audit
4. **04_primary_diff_in_diff_analysis.ipynb** -- corrected symbol-month DiD (turnover, expiry-day turnover, expiry-day share)
5. **05_robustness_checks.ipynb** -- placebo, window sensitivity, substitution range, participant mix, futures market quality
6. **06_expiry_volume_event_study.ipynb** -- expiry-day concentration for six NSE/BSE indices
7. **07_walk_forward_predictability.ipynb** -- secondary: leakage-free predictability test on the true calendar (null)
8. **08_interrupted_time_series.ipynb** -- secondary: settlement-weekday ITS plus clean weekday-only contrasts (null)
9. **09_figures.ipynb** -- displays the corrected figures
10. **10_option_microstructure_and_placebo.ipynb** -- option-level outcomes, stock-F&O placebo, dose-response

**Current location:** **04_primary_diff_in_diff_analysis.ipynb** <- *you are here*


In [1]:
import sys
from pathlib import Path

project_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

import pandas as pd
import json
from src import config

pd.set_option('display.max_columns', 20)
print('Project root:', project_dir)

Project root: E:\Research-IPR\Research_BN_N50\banknifty-expiry-volatility-ml


### 1. NSE only (Nifty control; Nifty is contaminated by substitution)
Outcomes are log points: `lt` total turnover, `le` expiry-day turnover, `ls` expiry-day share.

In [2]:
r = json.load(open(config.RESULTS_DIR / 'did_monthly_true_expiry.json'))
pd.DataFrame({k: v for k, v in r.items() if not k.startswith('eventstudy')}).T

,coef,se,p,n
BN vs NIFTY|lt,-2.821011,0.132738,3.125240e-100,161.0
BN vs NIFTY|le,-1.420718,0.144762,9.782962e-23,161.0
BN vs NIFTY|ls,1.400293,0.068026,3.769748e-94,161.0
BN+FIN+MIDCP vs NIFTY|lt,-4.076226,0.549141,1.145728e-13,285.0
BN+FIN+MIDCP vs NIFTY|le,-3.014595,0.564729,9.391041e-08,285.0
BN+FIN+MIDCP vs NIFTY|ls,1.061631,0.111840,2.257849e-21,285.0


### 2. Cross-exchange replication and pooled design
BSE retired Bankex weeklies but kept Sensex weeklies. The pooled estimate uses only 6 units, so inference is by exact randomization over treatment assignments (minimum attainable p = 1/15).

In [3]:
x = json.load(open(config.RESULTS_DIR / 'cross_exchange_did.json'))
within = {k: {o: round(v['coef'], 2) for o, v in x[k].items()} for k in x if ': ' in k}
print(pd.DataFrame(within).T)
print(pd.DataFrame(x['pooled_twfe']).T)

                           lt    le    ls
NSE: BANKNIFTY vs NIFTY -1.97 -0.62  1.35
BSE: BANKEX vs SENSEX   -4.84 -3.82  1.02
        coef  randomization_p_two_sided  n_assignments  rank_of_true_by_abs
lt -3.296496                   0.200000           15.0                  3.0
le -1.788674                   0.333333           15.0                  5.0
ls  1.507821                   0.066667           15.0                  1.0
